# ARC-AGI-3 Submit

In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:

import os
import sys
import gc
import re
import json
import time
import math
import copy
import heapq
import random
import types
import inspect
import logging
import hashlib
import traceback
import threading
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from collections import Counter, deque, defaultdict
from itertools import permutations

import numpy as np
import pandas as pd
import requests

from arc_agi import Arcade, OperationMode
from arcengine import ActionInput, GameAction, GameState


warnings.filterwarnings("ignore")
random.seed(0)
np.random.seed(0)

START_TIME = time.time()
TOTAL_BUDGET_SECONDS = int(5.45 * 3600)
SEARCH_BUDGET_SECONDS = int(5.05 * 3600)
SERVER_PORT = 8001
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

INPUT_ROOTS = [
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3"),
    Path("/kaggle/input/arc-prize-2026-arc-agi-3"),
]
COMP_ROOT = None
for p in INPUT_ROOTS:
    if p.exists():
        COMP_ROOT = p
        break
if COMP_ROOT is None:
    raise FileNotFoundError("ARC-AGI-3 competition input folder not found.")

ENV_DIR = COMP_ROOT / "environment_files"
if not ENV_DIR.exists():
    raise FileNotFoundError(f"Missing environment_files: {ENV_DIR}")

logger = logging.getLogger("arc_submit")
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(handler)

ALL_SIMPLE = [name for name in ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5", "ACTION7"] if hasattr(GameAction, name)]
HAS_COMPLEX = hasattr(GameAction, "ACTION6")
ACTOR_WORDS = ("player", "agent", "hero", "avatar", "cursor", "robot", "goose")
TARGET_WORDS = ("goal", "target", "exit", "finish", "home", "portal", "door", "box", "crate", "key", "gem", "food", "item")
RELEVANT_WORDS = ACTOR_WORDS + TARGET_WORDS + ("selected", "count", "mode", "phase", "toggle", "score", "level", "pos", "coord")
EPHEMERAL = {
    "_action_count", "_full_reset", "_action_complete",
    "_recording_file", "_recording_fp", "_recording_path",
    "_renderer", "renderer", "logger", "_scorecard_manager", "scorecard_manager",
    "_session", "_lock", "_cookie_lock", "_master_cookie_jar",
    "_api", "_last_response", "_recording_enabled",
}


@dataclass(frozen=True)
class ActionSpec:
    action_name: str
    x: int | None = None
    y: int | None = None

    @property
    def is_complex(self) -> bool:
        return self.x is not None and self.y is not None

    def enum(self):
        return getattr(GameAction, self.action_name)

    def input(self):
        if self.is_complex:
            return ActionInput(id=self.enum(), data={"x": int(self.x), "y": int(self.y)})
        return ActionInput(id=self.enum())

    def payload(self):
        return {"x": int(self.x), "y": int(self.y)} if self.is_complex else None

    def key(self):
        return (self.action_name, self.x, self.y)


@dataclass
class CompareHint:
    left_kind: str
    left_attr: str
    op: str
    right_kind: str
    right_value: object
    condition_line: str


@dataclass
class GameInfo:
    game_id: str
    class_name: str
    title: str
    local_dir: Path
    metadata: dict
    source_path: Path
    source_code: str


@dataclass
class LevelModel:
    bg: int
    simple_actions: list
    click_actions: list
    latent_clicks: list
    hidden_keys: list
    compare_hints: list
    action_bias: dict
    opposites: dict
    warmup_prefix: list = field(default_factory=list)


def remaining_total_time():
    return max(0.0, TOTAL_BUDGET_SECONDS - (time.time() - START_TIME))


def norm_action_name(obj):
    if obj is None:
        return None
    if isinstance(obj, str):
        name = obj.split(".")[-1]
        return name if hasattr(GameAction, name) else None
    if hasattr(obj, "name"):
        name = str(getattr(obj, "name"))
        if hasattr(GameAction, name):
            return name
    for attr in ("id", "value"):
        if hasattr(obj, attr):
            try:
                val = int(getattr(obj, attr))
                if hasattr(GameAction, "from_id"):
                    return GameAction.from_id(val).name
            except Exception:
                pass
    try:
        val = int(obj)
        if hasattr(GameAction, "from_id"):
            return GameAction.from_id(val).name
    except Exception:
        pass
    return None


def get_frame(game):
    return np.asarray(game.get_pixels(0, 0, 64, 64), dtype=np.uint8)


def infer_bg(frame):
    try:
        border = np.concatenate([frame[0], frame[-1], frame[:, 0], frame[:, -1]])
        vals = np.bincount(border.astype(np.int64), minlength=16)
        if vals.sum():
            return int(vals.argmax())
    except Exception:
        pass
    vals = np.bincount(frame.astype(np.int64).ravel(), minlength=16)
    return int(vals.argmax()) if len(vals) else 0


def get_state(resp):
    return getattr(resp, "state", None)


def is_game_over(resp):
    state = get_state(resp)
    try:
        return state == GameState.GAME_OVER or state is GameState.GAME_OVER
    except Exception:
        return str(state).endswith("GAME_OVER")


def is_final_win(resp):
    state = get_state(resp)
    try:
        return state == GameState.WIN or state is GameState.WIN
    except Exception:
        return str(state).endswith("WIN")


def get_levels_completed(resp, game=None):
    for name in ("levels_completed", "score"):
        if hasattr(resp, name):
            try:
                return int(getattr(resp, name))
            except Exception:
                pass
    if game is not None:
        for name in ("_score", "score", "levels_completed"):
            if hasattr(game, name):
                try:
                    return int(getattr(game, name))
                except Exception:
                    pass
    return 0


def get_level_index(game=None, resp=None):
    if game is not None:
        for name in ("_current_level_index", "current_level", "level_index", "level"):
            if hasattr(game, name):
                try:
                    return int(getattr(game, name))
                except Exception:
                    pass
    if resp is not None:
        for name in ("current_level_index", "level_index", "level"):
            if hasattr(resp, name):
                try:
                    return int(getattr(resp, name))
                except Exception:
                    pass
    return 0


def baseline(game, resp):
    return get_level_index(game, resp), get_levels_completed(resp, game)


def solved_from(base_level_idx, base_completed, game, resp):
    if get_levels_completed(resp, game) > base_completed:
        return True
    if get_level_index(game, resp) > base_level_idx:
        return True
    return False


def perform(game, act: ActionSpec):
    return game.perform_action(act.input(), raw=True)


def reset_game(game):
    return game.perform_action(ActionInput(id=GameAction.RESET), raw=True)


def safe_hash_val(v, depth=0, seen=None):
    if seen is None:
        seen = set()
    if depth > 3:
        return f"<{type(v).__name__}>"
    if v is None or isinstance(v, (bool, int, float, str)):
        return v
    if isinstance(v, np.generic):
        return v.item()
    oid = id(v)
    if oid in seen:
        return "<cycle>"
    seen.add(oid)
    if isinstance(v, np.ndarray):
        if v.ndim <= 2 and v.size <= 64:
            return ("nd", tuple(map(int, v.shape)), tuple(map(int, v.flatten().tolist())))
        return ("ndh", tuple(map(int, v.shape)), hashlib.blake2b(v.astype(np.int16, copy=False).tobytes(), digest_size=8).hexdigest())
    if isinstance(v, (tuple, list)):
        return tuple(safe_hash_val(x, depth + 1, seen) for x in list(v)[:16])
    if isinstance(v, dict):
        return tuple((str(k), safe_hash_val(v[k], depth + 1, seen)) for k in list(sorted(v.keys(), key=lambda x: str(x)))[:16])
    if hasattr(v, "x") and hasattr(v, "y"):
        try:
            return (type(v).__name__, int(getattr(v, "x")), int(getattr(v, "y")))
        except Exception:
            pass
    if hasattr(v, "__dict__"):
        items = []
        for k, val in list(sorted(v.__dict__.items()))[:16]:
            if k in EPHEMERAL:
                continue
            items.append((k, safe_hash_val(val, depth + 1, seen)))
        return (type(v).__name__, tuple(items))
    return repr(v)[:120]


def compact_hash(v):
    try:
        s = json.dumps(safe_hash_val(v), sort_keys=True, default=str)
    except Exception:
        s = repr(safe_hash_val(v))
    return hashlib.blake2b(s.encode("utf-8"), digest_size=8).hexdigest()


def frame_hash(frame):
    return hashlib.blake2b(frame.astype(np.uint8, copy=False).tobytes(), digest_size=12).hexdigest()


def state_fp(game, frame, keys):
    parts = [frame_hash(frame)]
    for k in keys:
        try:
            parts.append(f"{k}={compact_hash(getattr(game, k, None))}")
        except Exception:
            pass
    return "|".join(parts)


def skip_key(k, v):
    if k in EPHEMERAL:
        return True
    if k.startswith("__"):
        return True
    low = k.lower()
    if "logger" in low or "record" in low or "render" in low or "scorecard" in low:
        return True
    if inspect.ismodule(v) or inspect.isfunction(v) or inspect.ismethod(v) or inspect.isclass(v):
        return True
    return False


def is_int_like(x):
    try:
        int(x)
        return True
    except Exception:
        return False


def maybe_pos(obj):
    if isinstance(obj, np.ndarray) and obj.ndim == 1 and obj.size == 2:
        try:
            x, y = map(int, obj.tolist())
            if 0 <= x < 64 and 0 <= y < 64:
                return (x, y)
        except Exception:
            return None
    if isinstance(obj, (tuple, list)) and len(obj) == 2 and all(is_int_like(v) for v in obj):
        x, y = map(int, obj)
        if 0 <= x < 64 and 0 <= y < 64:
            return (x, y)
    if hasattr(obj, "x") and hasattr(obj, "y"):
        try:
            x = int(getattr(obj, "x"))
            y = int(getattr(obj, "y"))
            if 0 <= x < 64 and 0 <= y < 64:
                return (x, y)
        except Exception:
            return None
    return None


def extract_positions(obj, depth=0, seen=None):
    if seen is None:
        seen = set()
    if obj is None or depth > 3:
        return []
    oid = id(obj)
    if oid in seen:
        return []
    seen.add(oid)
    pt = maybe_pos(obj)
    if pt is not None:
        return [pt]
    out = []
    if isinstance(obj, np.ndarray):
        if obj.ndim == 2 and obj.shape[1] == 2 and obj.shape[0] <= 32:
            for row in obj:
                p = maybe_pos(row)
                if p is not None:
                    out.append(p)
        return out
    if isinstance(obj, (tuple, list)):
        if len(obj) <= 32:
            for item in obj:
                out.extend(extract_positions(item, depth + 1, seen))
        return out
    if isinstance(obj, dict):
        if len(obj) <= 32:
            for item in obj.values():
                out.extend(extract_positions(item, depth + 1, seen))
        return out
    if hasattr(obj, "__dict__"):
        for k, v in list(obj.__dict__.items())[:32]:
            if skip_key(k, v):
                continue
            out.extend(extract_positions(v, depth + 1, seen))
    return out


def named_positions(game):
    out = defaultdict(list)
    if not hasattr(game, "__dict__"):
        return out
    for k, v in game.__dict__.items():
        if skip_key(k, v):
            continue
        pts = extract_positions(v, depth=0, seen=set())
        if pts:
            seen = set()
            uniq = []
            for pt in pts:
                if pt not in seen:
                    seen.add(pt)
                    uniq.append(pt)
            out[k].extend(uniq[:16])
    return out


def connected_components(frame, bg):
    h, w = frame.shape
    seen = np.zeros((h, w), dtype=bool)
    comps = []
    for y in range(h):
        for x in range(w):
            if seen[y, x] or int(frame[y, x]) == bg:
                continue
            color = int(frame[y, x])
            stack = [(x, y)]
            seen[y, x] = True
            cells = []
            while stack:
                cx, cy = stack.pop()
                cells.append((cx, cy))
                if cx > 0 and (not seen[cy, cx - 1]) and int(frame[cy, cx - 1]) == color:
                    seen[cy, cx - 1] = True
                    stack.append((cx - 1, cy))
                if cx + 1 < w and (not seen[cy, cx + 1]) and int(frame[cy, cx + 1]) == color:
                    seen[cy, cx + 1] = True
                    stack.append((cx + 1, cy))
                if cy > 0 and (not seen[cy - 1, cx]) and int(frame[cy - 1, cx]) == color:
                    seen[cy - 1, cx] = True
                    stack.append((cx, cy - 1))
                if cy + 1 < h and (not seen[cy + 1, cx]) and int(frame[cy + 1, cx]) == color:
                    seen[cy + 1, cx] = True
                    stack.append((cx, cy + 1))
            xs = [p[0] for p in cells]
            ys = [p[1] for p in cells]
            comps.append({
                "color": color,
                "size": len(cells),
                "cells": cells,
                "x0": min(xs), "x1": max(xs),
                "y0": min(ys), "y1": max(ys),
                "cx": int(round(sum(xs) / len(xs))),
                "cy": int(round(sum(ys) / len(ys))),
            })
    comps.sort(key=lambda c: (-c["size"], c["y0"], c["x0"], c["color"]))
    return comps


def bbox_non_bg(frame, bg):
    ys, xs = np.where(frame != bg)
    if len(xs) == 0:
        return (0, 0, frame.shape[1] - 1, frame.shape[0] - 1)
    return (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))


def clip_xy(x, y):
    return max(0, min(63, int(x))), max(0, min(63, int(y)))


def parse_win_hints(source):
    lines = source.splitlines()
    conds = []
    for i, line in enumerate(lines):
        if "self.next_level(" in line or ("GameState.WIN" in line and "=" in line):
            for j in range(i - 1, max(-1, i - 12), -1):
                s = lines[j].strip()
                if s.startswith("if ") or s.startswith("elif "):
                    conds.append(s.rstrip(":"))
                    break
    attrs = []
    def add_attr(a):
        if a and a not in attrs:
            attrs.append(a)
    for cond in conds:
        for a in re.findall(r"self\.(\w+)", cond):
            add_attr(a)

    hints = []
    pats = [
        re.compile(r"len\(self\.(\w+)\)\s*(==|!=|>=|<=|>|<)\s*(-?\d+)"),
        re.compile(r"self\.(\w+)\s*(==|!=|>=|<=|>|<)\s*len\(self\.(\w+)\)"),
        re.compile(r"self\.(\w+)\s*(==|!=|>=|<=|>|<)\s*(True|False|None|-?\d+)"),
        re.compile(r"self\.(\w+)\s*(==|!=|>=|<=|>|<)\s*self\.(\w+)"),
    ]
    for cond in conds:
        for m in pats[0].finditer(cond):
            a, op, c = m.groups()
            hints.append(CompareHint("len", a, op, "const", int(c), cond))
        for m in pats[1].finditer(cond):
            a, op, b = m.groups()
            hints.append(CompareHint("attr", a, op, "len_attr", b, cond))
        for m in pats[2].finditer(cond):
            a, op, raw = m.groups()
            if raw == "True":
                rv = True
            elif raw == "False":
                rv = False
            elif raw == "None":
                rv = None
            else:
                rv = int(raw)
            hints.append(CompareHint("attr", a, op, "const", rv, cond))
        for m in pats[3].finditer(cond):
            a, op, b = m.groups()
            hints.append(CompareHint("attr", a, op, "attr", b, cond))
    return attrs, hints


def hint_values(game, hint):
    try:
        lv = getattr(game, hint.left_attr, None)
    except Exception:
        lv = None
    if hint.left_kind == "len":
        try:
            lv = len(lv)
        except Exception:
            lv = None
    if hint.right_kind == "const":
        rv = hint.right_value
    elif hint.right_kind == "attr":
        rv = getattr(game, hint.right_value, None)
    elif hint.right_kind == "len_attr":
        try:
            rv = len(getattr(game, hint.right_value, None))
        except Exception:
            rv = None
    else:
        rv = None
    return lv, rv


def compare_distance(game, hint):
    lv, rv = hint_values(game, hint)
    lvp = maybe_pos(lv)
    rvp = maybe_pos(rv)
    if lvp is not None and rvp is not None:
        d = abs(lvp[0] - rvp[0]) + abs(lvp[1] - rvp[1])
        if hint.op == "==":
            return float(d)
        if hint.op == "!=":
            return 0.0 if d > 0 else 1.0
        return float(d)
    if isinstance(lv, (int, float, bool)) and isinstance(rv, (int, float, bool)):
        lv = float(lv)
        rv = float(rv)
        if hint.op == "==":
            return abs(lv - rv)
        if hint.op == "!=":
            return 0.0 if lv != rv else 1.0
        if hint.op in (">=", ">"):
            return max(0.0, rv - lv)
        if hint.op in ("<=", "<"):
            return max(0.0, lv - rv)
    return None


def progress_value(game, resp, base_level_idx, base_completed, hints):
    val = 100000.0 if solved_from(base_level_idx, base_completed, game, resp) else 0.0
    for hint in hints:
        d = compare_distance(game, hint)
        if d is not None:
            val += max(0.0, 100.0 - min(100.0, float(d)))
    return val


def available_actions(game, resp=None):
    raw = []
    if resp is not None and hasattr(resp, "available_actions"):
        raw = list(getattr(resp, "available_actions") or [])
    if not raw and hasattr(game, "_available_actions"):
        raw = list(getattr(game, "_available_actions") or [])
    names = []
    for obj in raw:
        name = norm_action_name(obj)
        if name is not None:
            names.append(name)
    simple = [ActionSpec(name) for name in ALL_SIMPLE if name in names]
    has_complex = HAS_COMPLEX and ("ACTION6" in names)
    if not simple and not has_complex:
        simple = [ActionSpec(name) for name in ALL_SIMPLE]
        has_complex = HAS_COMPLEX
    return simple, has_complex


def detect_actor_delta(game0, game1, frame0, frame1, bg):
    pos0 = named_positions(game0)
    pos1 = named_positions(game1)
    for k in pos0.keys():
        low = k.lower()
        if not any(w in low for w in ACTOR_WORDS):
            continue
        if len(pos0[k]) == 1 and len(pos1.get(k, [])) == 1:
            x0, y0 = pos0[k][0]
            x1, y1 = pos1[k][0]
            if (x0, y0) != (x1, y1):
                dx, dy = x1 - x0, y1 - y0
                if abs(dx) + abs(dy) <= 4:
                    return (dx, dy)
    diff = frame0 != frame1
    if not np.any(diff):
        return None
    lost = np.argwhere(diff & (frame0 != bg))
    gained = np.argwhere(diff & (frame1 != bg))
    if 0 < len(lost) <= 64 and 0 < len(gained) <= 64:
        ly, lx = lost.mean(axis=0)
        gy, gx = gained.mean(axis=0)
        dx = int(round(gx - lx))
        dy = int(round(gy - ly))
        if abs(dx) + abs(dy) <= 4:
            return (dx, dy)
    return None


def game_candidate_keys(game):
    keys = []
    if hasattr(game, "__dict__"):
        for k, v in game.__dict__.items():
            if skip_key(k, v):
                continue
            if isinstance(v, (bool, int, float, str, tuple, list, dict, np.ndarray)) or maybe_pos(v) is not None:
                keys.append(k)
            elif any(w in k.lower() for w in RELEVANT_WORDS):
                keys.append(k)
    return list(dict.fromkeys(keys))


def rank_clicks(game, frame, bg, max_points=40, extra=None, cache=None):
    fh = frame_hash(frame)
    if cache is not None and fh in cache:
        pts = cache[fh]
        if extra:
            seen = set(pts)
            merged = list(pts)
            for pt in extra:
                if pt not in seen:
                    merged.append(pt)
                    seen.add(pt)
            return merged[:max_points]
        return pts[:max_points]

    scores = defaultdict(float)
    def add(pt, score):
        if pt is None:
            return
        x, y = clip_xy(pt[0], pt[1])
        if 0 <= x < frame.shape[1] and 0 <= y < frame.shape[0]:
            scores[(x, y)] = max(scores[(x, y)], float(score))

    for k, pts in named_positions(game).items():
        base = 60.0
        low = k.lower()
        if any(w in low for w in ACTOR_WORDS):
            base = 75.0
        if any(w in low for w in TARGET_WORDS):
            base = 85.0
        for pt in pts[:6]:
            add(pt, base)
            x, y = pt
            for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
                add((x + dx, y + dy), base - 10.0)

    comps = connected_components(frame, bg)
    color_counts = Counter(frame.ravel().tolist())
    for comp in comps[:20]:
        rarity = 1.0 / max(1, color_counts[comp["color"]])
        base = 45.0 + 30.0 * rarity + max(0.0, 12.0 - math.log2(comp["size"] + 1.0))
        add((comp["cx"], comp["cy"]), base + 3.0)
        add((comp["x0"], comp["y0"]), base)
        add((comp["x1"], comp["y0"]), base)
        add((comp["x0"], comp["y1"]), base)
        add((comp["x1"], comp["y1"]), base)
        add(((comp["x0"] + comp["x1"]) // 2, comp["cy"]), base - 2.0)
        add((comp["cx"], (comp["y0"] + comp["y1"]) // 2), base - 2.0)

    x0, y0, x1, y1 = bbox_non_bg(frame, bg)
    add((x0, y0), 20.0); add((x1, y0), 20.0); add((x0, y1), 20.0); add((x1, y1), 20.0)
    add(((x0 + x1) // 2, (y0 + y1) // 2), 18.0)
    add((32, 32), 8.0)

    ys, xs = np.where(frame != bg)
    if len(xs):
        step = max(1, len(xs) // 24)
        for i in range(0, len(xs), step):
            add((int(xs[i]), int(ys[i])), 10.0)

    if extra:
        for rank, pt in enumerate(extra):
            add(pt, 25.0 - 0.25 * rank)

    pts = [pt for pt, _ in sorted(scores.items(), key=lambda kv: (-kv[1], kv[0][1], kv[0][0]))]
    if cache is not None:
        cache[fh] = pts[:64]
    return pts[:max_points]


def source_class_name(source, fallback):
    m = re.search(r"class\s+(\w+)\s*\(\s*ARCBaseGame", source)
    if m:
        return m.group(1)
    m = re.search(r"class\s+(\w+)\s*\(", source)
    if m:
        return m.group(1)
    return fallback


def choose_source_path(local_dir, metadata):
    gid = str(metadata.get("game_id") or metadata.get("id") or local_dir.name)
    class_name = metadata.get("class_name") or metadata.get("game_class_name") or metadata.get("environment_class_name")
    candidates = []
    for name in [f"{gid}.py", f"{str(class_name).lower()}.py" if class_name else None]:
        if name:
            p = local_dir / name
            if p.exists():
                candidates.append(p)
    if not candidates:
        py_files = [p for p in local_dir.glob("*.py") if p.name != "__init__.py"]
        py_files.sort(key=lambda p: (p.name != f"{gid}.py", p.name))
        if py_files:
            candidates.append(py_files[0])
    return candidates[0] if candidates else None


def scan_games(env_dir):
    infos = []
    for meta_path in sorted(env_dir.rglob("metadata.json")):
        try:
            metadata = json.loads(meta_path.read_text())
        except Exception:
            continue
        local_dir = meta_path.parent
        gid = str(metadata.get("game_id") or metadata.get("id") or local_dir.name)
        title = str(metadata.get("title") or gid)
        source_path = choose_source_path(local_dir, metadata)
        if source_path is None or not source_path.exists():
            continue
        try:
            source = source_path.read_text()
        except Exception:
            continue
        class_name = metadata.get("class_name") or metadata.get("game_class_name") or metadata.get("environment_class_name")
        class_name = source_class_name(source, class_name or gid.capitalize())
        infos.append(GameInfo(
            game_id=gid,
            class_name=class_name,
            title=title,
            local_dir=local_dir,
            metadata=metadata,
            source_path=source_path,
            source_code=source,
        ))
    seen = set()
    uniq = []
    for info in infos:
        if info.game_id in seen:
            continue
        seen.add(info.game_id)
        uniq.append(info)
    uniq.sort(key=lambda info: ("ACTION6" in info.source_code, len(info.source_code), info.game_id))
    return uniq


class GameLoader:
    def __init__(self, info: GameInfo):
        self.info = info
        self.cls = None
        self.accepts_seed = False

    def load(self):
        if self.cls is not None:
            return self.cls
        module_name = f"arc_submit_{self.info.game_id}_{abs(hash(str(self.info.source_path)))}"
        old_path = list(sys.path)
        try:
            sys.path.insert(0, str(self.info.local_dir))
            module = types.ModuleType(module_name)
            module.__file__ = str(self.info.source_path)
            sys.modules[module_name] = module
            code = compile(self.info.source_code, str(self.info.source_path), "exec")
            exec(code, module.__dict__)
            self.cls = getattr(module, self.info.class_name)
            try:
                self.accepts_seed = "seed" in inspect.signature(self.cls).parameters
            except Exception:
                self.accepts_seed = False
            return self.cls
        finally:
            sys.path[:] = old_path

    def new_game(self, seed=0):
        cls = self.load()
        if self.accepts_seed:
            try:
                return cls(seed=seed)
            except Exception:
                pass
        return cls()


class Solver:
    def __init__(self, info: GameInfo, loader: GameLoader, deadline: float, scan_timeout: float = 2.0):
        self.info = info
        self.loader = loader
        self.deadline = deadline
        self.scan_timeout = scan_timeout
        self.solutions = {}
        self.click_cache = {}

    def time_left(self):
        return max(0.0, self.deadline - time.time())

    def fresh_level(self, level_idx):
        g = self.loader.new_game(seed=0)
        if hasattr(g, "set_level"):
            g.set_level(int(level_idx))
        resp = None
        for _ in range(3):
            try:
                resp = reset_game(g)
            except Exception:
                pass
        if resp is None:
            raise RuntimeError("RESET failed")
        return g, resp

    def sequence_solves(self, level_idx, seq):
        try:
            g, resp0 = self.fresh_level(level_idx)
        except Exception:
            return False
        base_level_idx, base_completed = baseline(g, resp0)
        for act in seq:
            try:
                resp = perform(g, act)
            except Exception:
                return False
            if solved_from(base_level_idx, base_completed, g, resp):
                return True
            if is_game_over(resp):
                return False
        return False

    def reduce_solution(self, level_idx, seq, max_time=3.0):
        if len(seq) <= 2:
            return list(seq)
        end = min(self.deadline, time.time() + max_time)
        best = list(seq)
        improved = True
        while improved and time.time() < end:
            improved = False
            chunk_sizes = sorted(set([max(1, len(best) // 2), max(1, len(best) // 3), 3, 2, 1]), reverse=True)
            for chunk in chunk_sizes:
                i = 0
                while i + chunk <= len(best) and time.time() < end:
                    cand = best[:i] + best[i + chunk:]
                    if cand and self.sequence_solves(level_idx, cand):
                        best = cand
                        improved = True
                    else:
                        i += 1
                if improved:
                    break
        return best

    def scan_actions(self, game, resp0, frame0, bg):
        base_level_idx, base_completed = baseline(game, resp0)
        candidate_keys = game_candidate_keys(game)[:32]
        base_vals = {k: compact_hash(getattr(game, k, None)) for k in candidate_keys}
        win_attrs, compare_hints = parse_win_hints(self.info.source_code)
        simple_actions, has_complex = available_actions(game, resp0)

        effective_simple = []
        click_actions = []
        latent_clicks = []
        action_bias = {}
        probe_records = []

        for act in simple_actions:
            g = copy.deepcopy(game)
            try:
                r = perform(g, act)
            except Exception:
                continue
            f = get_frame(g)
            pixels = int(np.count_nonzero(frame0 != f))
            changed = []
            for k in candidate_keys:
                try:
                    hv = compact_hash(getattr(g, k, None))
                except Exception:
                    continue
                if hv != base_vals.get(k):
                    changed.append(k)
            delta = detect_actor_delta(game, g, frame0, f, bg)
            adv = solved_from(base_level_idx, base_completed, g, r)
            if pixels > 0 or changed:
                effective_simple.append(act)
                bias = 0.02 * pixels + 2.0 * len(changed)
                if delta is not None:
                    bias += 8.0
                if adv:
                    bias += 1000.0
                action_bias[act] = bias
            probe_records.append((act, changed, delta))

        if has_complex:
            positions = rank_clicks(game, frame0, bg, max_points=48, cache=self.click_cache)
            t0 = time.time()
            for rank, (x, y) in enumerate(positions):
                if time.time() - t0 > min(self.scan_timeout, max(0.6, self.time_left() * 0.15)):
                    break
                act = ActionSpec("ACTION6", x, y)
                g = copy.deepcopy(game)
                try:
                    r = perform(g, act)
                except Exception:
                    continue
                f = get_frame(g)
                pixels = int(np.count_nonzero(frame0 != f))
                changed = []
                for k in candidate_keys:
                    try:
                        hv = compact_hash(getattr(g, k, None))
                    except Exception:
                        continue
                    if hv != base_vals.get(k):
                        changed.append(k)
                if pixels > 0 or changed:
                    click_actions.append(act)
                    action_bias[act] = 0.02 * pixels + 2.0 * len(changed) + max(0.0, 20.0 - 0.2 * rank)
                    probe_records.append((act, changed, None))
                elif rank < 8:
                    latent_clicks.append(act)

        hidden_keys = []
        for name in ("_current_level_index", "_score"):
            if hasattr(game, name):
                hidden_keys.append(name)
        for a in win_attrs:
            if hasattr(game, a) and a not in hidden_keys:
                hidden_keys.append(a)
        for k in candidate_keys:
            if any(w in k.lower() for w in RELEVANT_WORDS) and k not in hidden_keys:
                hidden_keys.append(k)
        for _, changed, _ in probe_records:
            for k in changed:
                if k not in hidden_keys:
                    hidden_keys.append(k)
        hidden_keys = hidden_keys[:20]

        opposites = {}
        deltas = [(act.action_name, delta) for act, _, delta in probe_records if delta is not None and not act.is_complex]
        for a_name, a_delta in deltas:
            for b_name, b_delta in deltas:
                if a_name != b_name and a_delta[0] == -b_delta[0] and a_delta[1] == -b_delta[1]:
                    opposites[a_name] = b_name

        if has_complex and not click_actions and latent_clicks:
            click_actions = latent_clicks[:6]

        return LevelModel(
            bg=bg,
            simple_actions=effective_simple or simple_actions,
            click_actions=click_actions[:24],
            latent_clicks=latent_clicks[:8],
            hidden_keys=hidden_keys,
            compare_hints=compare_hints,
            action_bias=action_bias,
            opposites=opposites,
        )

    def warmup_unlock(self, level_idx, simple_actions):
        simple = [a for a in simple_actions if not a.is_complex][:4]
        if not simple:
            return None
        try:
            game, resp0 = self.fresh_level(level_idx)
        except Exception:
            return None
        base_level_idx, base_completed = baseline(game, resp0)
        frame0 = get_frame(game)
        for a in simple:
            for b in simple:
                g = copy.deepcopy(game)
                prefix = []
                try:
                    r1 = perform(g, a)
                    prefix.append(a)
                    f1 = get_frame(g)
                    if solved_from(base_level_idx, base_completed, g, r1):
                        return g, r1, f1, prefix, True
                    if np.count_nonzero(frame0 != f1) > 0:
                        return g, r1, f1, prefix, False
                    r2 = perform(g, b)
                    prefix.append(b)
                    f2 = get_frame(g)
                    if solved_from(base_level_idx, base_completed, g, r2):
                        return g, r2, f2, prefix, True
                    if np.count_nonzero(frame0 != f2) > 0:
                        return g, r2, f2, prefix, False
                except Exception:
                    continue
        return None

    def transfer_bbox(self, seq, prev_frame, cur_frame):
        bg0 = infer_bg(prev_frame)
        bg1 = infer_bg(cur_frame)
        ax0, ay0, ax1, ay1 = bbox_non_bg(prev_frame, bg0)
        bx0, by0, bx1, by1 = bbox_non_bg(cur_frame, bg1)
        wa = max(1, ax1 - ax0)
        ha = max(1, ay1 - ay0)
        wb = max(1, bx1 - bx0)
        hb = max(1, by1 - by0)
        out = []
        for act in seq:
            if act.is_complex:
                rx = (act.x - ax0) / wa
                ry = (act.y - ay0) / ha
                x = bx0 + int(round(rx * wb))
                y = by0 + int(round(ry * hb))
                x, y = clip_xy(x, y)
                out.append(ActionSpec("ACTION6", x, y))
            else:
                out.append(act)
        return out

    def transfer_components(self, seq, prev_frame, cur_frame):
        bg0 = infer_bg(prev_frame)
        bg1 = infer_bg(cur_frame)
        comps0 = connected_components(prev_frame, bg0)
        comps1 = connected_components(cur_frame, bg1)
        by_color0 = defaultdict(list)
        by_color1 = defaultdict(list)
        for comp in comps0:
            by_color0[comp["color"]].append(comp)
        for comp in comps1:
            by_color1[comp["color"]].append(comp)

        def sig(frame, by_color, x, y):
            color = int(frame[y, x])
            arr = by_color.get(color, [])
            for idx, comp in enumerate(arr):
                if comp["x0"] <= x <= comp["x1"] and comp["y0"] <= y <= comp["y1"]:
                    return color, idx, x - comp["x0"], y - comp["y0"]
            if arr:
                comp = min(arr, key=lambda c: abs(c["cx"] - x) + abs(c["cy"] - y))
                idx = arr.index(comp)
                return color, idx, x - comp["x0"], y - comp["y0"]
            return color, 0, 0, 0

        out = []
        for act in seq:
            if not act.is_complex:
                out.append(act)
                continue
            color, idx, dx, dy = sig(prev_frame, by_color0, act.x, act.y)
            arr = by_color1.get(color, [])
            if not arr:
                out.append(act)
                continue
            comp = arr[min(idx, len(arr) - 1)]
            x = max(comp["x0"], min(comp["x1"], comp["x0"] + dx))
            y = max(comp["y0"], min(comp["y1"], comp["y0"] + dy))
            out.append(ActionSpec("ACTION6", x, y))
        return out

    def transfer_candidates(self, level_idx, prev_solution, current_frame):
        if not prev_solution or level_idx <= 0:
            return []
        out = [list(prev_solution)]
        if any(a.is_complex for a in prev_solution):
            try:
                pg, _ = self.fresh_level(level_idx - 1)
                prev_frame = get_frame(pg)
                out.append(self.transfer_bbox(prev_solution, prev_frame, current_frame))
                out.append(self.transfer_components(prev_solution, prev_frame, current_frame))
                for mult in (2, 3):
                    expanded = []
                    for act in prev_solution:
                        expanded.extend([act] * mult)
                    out.append(expanded)
            except Exception:
                pass
        uniq = []
        seen = set()
        for seq in out:
            key = tuple(a.key() for a in seq)
            if key not in seen:
                seen.add(key)
                uniq.append(seq)
        return uniq

    def try_transfer(self, level_idx, prev_solution, current_frame):
        for seq in self.transfer_candidates(level_idx, prev_solution, current_frame):
            try:
                game, resp0 = self.fresh_level(level_idx)
            except Exception:
                break
            base_level_idx, base_completed = baseline(game, resp0)
            for i, act in enumerate(seq):
                try:
                    resp = perform(game, act)
                except Exception:
                    break
                if solved_from(base_level_idx, base_completed, game, resp):
                    return self.reduce_solution(level_idx, seq[:i + 1], max_time=1.5)
                if is_game_over(resp):
                    break
        return None

    def component_perm_search(self, level_idx, model: LevelModel):
        clicks = []
        seen = set()
        for act in model.click_actions[:5] + model.latent_clicks[:3]:
            if act.key() not in seen:
                seen.add(act.key())
                clicks.append(act)
        if not clicks:
            return None
        tails = [[]] + [[a] for a in model.simple_actions[:2]]
        for length in range(1, min(4, len(clicks)) + 1):
            for perm in permutations(clicks, length):
                base = list(perm)
                for tail in tails:
                    seq = base + tail
                    if self.sequence_solves(level_idx, seq):
                        return self.reduce_solution(level_idx, seq, max_time=1.0)
        return None

    def ordered_actions(self, model: LevelModel, last_action=None, dynamic_clicks=None):
        scored = []
        for act in model.simple_actions:
            score = model.action_bias.get(act, 0.0)
            if last_action is not None and last_action.action_name == act.action_name:
                score += 2.0
            if last_action is not None and model.opposites.get(last_action.action_name) == act.action_name:
                score -= 10.0
            scored.append((score, act))
        clicks = dynamic_clicks if dynamic_clicks is not None else (model.click_actions + model.latent_clicks)
        for rank, act in enumerate(clicks):
            score = model.action_bias.get(act, 12.0 - 0.2 * rank)
            if last_action is not None and last_action.is_complex and act.is_complex and last_action.x == act.x and last_action.y == act.y:
                score -= 12.0
            scored.append((score, act))
        scored.sort(key=lambda t: (-t[0], t[1].action_name, t[1].x if t[1].x is not None else -1, t[1].y if t[1].y is not None else -1))
        out = []
        seen = set()
        for _, act in scored:
            if act.key() not in seen:
                seen.add(act.key())
                out.append(act)
        return out

    def dynamic_actions(self, game, frame, model: LevelModel, depth, last_action):
        extra = [(a.x, a.y) for a in model.click_actions + model.latent_clicks if a.is_complex]
        clicks = []
        if HAS_COMPLEX:
            coords = rank_clicks(game, frame, model.bg, max_points=(20 if depth <= 2 else 12), extra=extra, cache=self.click_cache)
            limit = 14 if depth <= 2 else 8
            clicks = [ActionSpec("ACTION6", x, y) for x, y in coords[:limit]]
        return self.ordered_actions(model, last_action, clicks)

    def bfs(self, level_idx, model: LevelModel, depth_limit=30):
        try:
            root_game, root_resp = self.fresh_level(level_idx)
        except Exception:
            return None
        base_level_idx, base_completed = baseline(root_game, root_resp)
        root_frame = get_frame(root_game)
        visited = {state_fp(root_game, root_frame, model.hidden_keys)}
        q = deque([(copy.deepcopy(root_game), root_frame, [], None)])
        explored = 0
        end = min(self.deadline, time.time() + max(2.0, self.time_left() * 0.55))
        while q and time.time() < end:
            game, frame, hist, last_action = q.popleft()
            if len(hist) >= depth_limit:
                continue
            for act in self.ordered_actions(model, last_action):
                g2 = copy.deepcopy(game)
                try:
                    resp = perform(g2, act)
                except Exception:
                    continue
                explored += 1
                f2 = get_frame(g2)
                fp = state_fp(g2, f2, model.hidden_keys)
                if fp in visited:
                    continue
                visited.add(fp)
                new_hist = hist + [act]
                if solved_from(base_level_idx, base_completed, g2, resp):
                    logger.info(f"{self.info.game_id} L{level_idx}: BFS solved in {len(new_hist)} actions ({explored} explored)")
                    return self.reduce_solution(level_idx, new_hist, max_time=2.0)
                if is_game_over(resp):
                    continue
                q.append((g2, f2, new_hist, act))
        logger.info(f"{self.info.game_id} L{level_idx}: BFS exhausted ({explored} explored, {len(visited)} unique)")
        return None

    def acmd(self, level_idx, model: LevelModel):
        if not model.hidden_keys:
            return None
        try:
            root_game, root_resp = self.fresh_level(level_idx)
        except Exception:
            return None
        base_level_idx, base_completed = baseline(root_game, root_resp)
        root_frame = get_frame(root_game)
        base_state = {k: getattr(root_game, k, None) for k in model.hidden_keys}
        visited = {state_fp(root_game, root_frame, model.hidden_keys)}
        heap = [(0.0, 0, 0, copy.deepcopy(root_game), root_frame, [], None)]
        counter = 1
        explored = 0
        end = min(self.deadline, time.time() + max(2.0, self.time_left() * 0.35))
        while heap and time.time() < end:
            _, depth, _, game, frame, hist, last_action = heapq.heappop(heap)
            if depth >= 40:
                continue
            for act in self.dynamic_actions(game, frame, model, depth, last_action):
                g2 = copy.deepcopy(game)
                try:
                    resp = perform(g2, act)
                except Exception:
                    continue
                explored += 1
                f2 = get_frame(g2)
                fp = state_fp(g2, f2, model.hidden_keys)
                if fp in visited:
                    continue
                visited.add(fp)
                new_hist = hist + [act]
                if solved_from(base_level_idx, base_completed, g2, resp):
                    logger.info(f"{self.info.game_id} L{level_idx}: ACMD solved in {len(new_hist)} actions ({explored} explored)")
                    return self.reduce_solution(level_idx, new_hist, max_time=2.0)
                if is_game_over(resp):
                    continue
                trigger_delta = 0.0
                for k in model.hidden_keys:
                    v0 = base_state.get(k, None)
                    v1 = getattr(g2, k, None)
                    if isinstance(v0, (int, float)) and isinstance(v1, (int, float)):
                        trigger_delta += abs(v1 - v0)
                    elif compact_hash(v0) != compact_hash(v1):
                        trigger_delta += 1.0
                pixel_delta = np.count_nonzero(root_frame != f2)
                if pixel_delta == 0 and trigger_delta == 0:
                    continue
                priority = -(trigger_delta * 10.0 + min(pixel_delta, 50) * 0.1) + depth
                heapq.heappush(heap, (priority, depth + 1, counter, g2, f2, new_hist, act))
                counter += 1
        logger.info(f"{self.info.game_id} L{level_idx}: ACMD exhausted ({explored} explored)")
        return None

    def heuristic(self, level_idx, model: LevelModel):
        try:
            root_game, root_resp = self.fresh_level(level_idx)
        except Exception:
            return None
        base_level_idx, base_completed = baseline(root_game, root_resp)
        root_frame = get_frame(root_game)
        root_fp = state_fp(root_game, root_frame, model.hidden_keys)
        best_depth = {root_fp: 0}
        root_prog = progress_value(root_game, root_resp, base_level_idx, base_completed, model.compare_hints)
        heap = [(0.0, 0, 0, copy.deepcopy(root_game), root_frame, [], None, root_prog)]
        counter = 1
        explored = 0
        best_prog = root_prog
        end = min(self.deadline, time.time() + max(2.0, self.time_left() * 0.6))
        while heap and time.time() < end:
            _, depth, _, game, frame, hist, last_action, _ = heapq.heappop(heap)
            if depth >= 48:
                continue
            for act in self.dynamic_actions(game, frame, model, depth, last_action):
                g2 = copy.deepcopy(game)
                try:
                    resp = perform(g2, act)
                except Exception:
                    continue
                explored += 1
                f2 = get_frame(g2)
                fp = state_fp(g2, f2, model.hidden_keys)
                new_depth = depth + 1
                if fp in best_depth and best_depth[fp] <= new_depth:
                    continue
                best_depth[fp] = new_depth
                new_hist = hist + [act]
                if solved_from(base_level_idx, base_completed, g2, resp):
                    logger.info(f"{self.info.game_id} L{level_idx}: heuristic solved in {len(new_hist)} actions ({explored} explored)")
                    return self.reduce_solution(level_idx, new_hist, max_time=2.0)
                if is_game_over(resp):
                    continue
                prog = progress_value(g2, resp, base_level_idx, base_completed, model.compare_hints)
                best_prog = max(best_prog, prog)
                dvals = [compare_distance(g2, hint) for hint in model.compare_hints]
                dvals = [d for d in dvals if d is not None]
                heuristic = sum(dvals) if dvals else 0.0
                priority = new_depth + 0.35 * heuristic - 0.015 * prog
                heapq.heappush(heap, (priority, new_depth, counter, g2, f2, new_hist, act, prog))
                counter += 1
        logger.info(f"{self.info.game_id} L{level_idx}: heuristic exhausted ({explored} explored, best_prog={best_prog:.2f})")
        return None

    def iddfs(self, level_idx, model: LevelModel, max_depth=60):
        if len(model.simple_actions) + len(model.click_actions) > 6:
            return None
        try:
            root_game, root_resp = self.fresh_level(level_idx)
        except Exception:
            return None
        base_level_idx, base_completed = baseline(root_game, root_resp)
        root_frame = get_frame(root_game)
        root_fp = state_fp(root_game, root_frame, model.hidden_keys)
        end = min(self.deadline, time.time() + max(2.0, self.time_left()))

        def dfs(game, frame, depth, limit, hist, last_action, path):
            if time.time() >= end:
                return None
            if depth >= limit:
                return None
            for act in self.ordered_actions(model, last_action):
                g2 = copy.deepcopy(game)
                try:
                    resp = perform(g2, act)
                except Exception:
                    continue
                f2 = get_frame(g2)
                fp = state_fp(g2, f2, model.hidden_keys)
                if fp in path:
                    continue
                new_hist = hist + [act]
                if solved_from(base_level_idx, base_completed, g2, resp):
                    return new_hist
                if is_game_over(resp):
                    continue
                out = dfs(g2, f2, depth + 1, limit, new_hist, act, path | {fp})
                if out is not None:
                    return out
            return None

        for limit in range(1, max_depth + 1):
            if time.time() >= end:
                break
            out = dfs(copy.deepcopy(root_game), root_frame, 0, limit, [], None, {root_fp})
            if out is not None:
                logger.info(f"{self.info.game_id} L{level_idx}: IDDFS solved at depth {limit}")
                return self.reduce_solution(level_idx, out, max_time=2.0)
        logger.info(f"{self.info.game_id} L{level_idx}: IDDFS exhausted")
        return None

    def solve_level(self, level_idx, prev_solution=None):
        try:
            game, resp0 = self.fresh_level(level_idx)
        except Exception as e:
            logger.warning(f"{self.info.game_id} L{level_idx}: fresh level failed: {e}")
            return None
        frame0 = get_frame(game)
        bg = infer_bg(frame0)

        if prev_solution:
            transferred = self.try_transfer(level_idx, prev_solution, frame0)
            if transferred:
                return transferred

        model = self.scan_actions(game, resp0, frame0, bg)
        if not model.simple_actions and not model.click_actions and not model.latent_clicks:
            warm = self.warmup_unlock(level_idx, available_actions(game, resp0)[0])
            if warm is not None:
                warm_game, warm_resp, warm_frame, prefix, solved = warm
                if solved:
                    return list(prefix)
                model = self.scan_actions(warm_game, warm_resp, warm_frame, infer_bg(warm_frame))
                model.warmup_prefix = prefix
            else:
                logger.info(f"{self.info.game_id} L{level_idx}: no effective actions")
                return None

        logger.info(
            f"{self.info.game_id} L{level_idx}: simple={len(model.simple_actions)} "
            f"click={len(model.click_actions)} latent={len(model.latent_clicks)} hidden={model.hidden_keys}"
        )

        strategies = [
            lambda: self.component_perm_search(level_idx, model),
            lambda: self.bfs(level_idx, model, depth_limit=30),
            lambda: self.acmd(level_idx, model),
            lambda: self.heuristic(level_idx, model),
            lambda: self.iddfs(level_idx, model, max_depth=60),
        ]
        for strategy in strategies:
            if self.time_left() <= 0:
                break
            sol = strategy()
            if sol:
                if model.warmup_prefix:
                    sol = list(model.warmup_prefix) + list(sol)
                return self.reduce_solution(level_idx, sol, max_time=2.0)
        return None

    def solve_game(self):
        plan = []
        solved_levels = 0
        prev_solution = None
        level_idx = 0
        while time.time() < self.deadline:
            sol = self.solve_level(level_idx, prev_solution)
            if not sol:
                break
            plan.extend(sol)
            self.solutions[level_idx] = sol
            prev_solution = sol
            solved_levels += 1
            level_idx += 1
        return plan, solved_levels


def start_local_server():
    local_arc = Arcade(
        operation_mode=OperationMode.OFFLINE,
        environments_dir=str(ENV_DIR),
        arc_api_key="local",
    )

    def serve():
        local_arc.listen_and_serve(
            host="127.0.0.1",
            port=SERVER_PORT,
            competition_mode=True,
            save_all_recordings=False,
            threaded=True,
            use_reloader=False,
        )

    thread = threading.Thread(target=serve, daemon=True)
    thread.start()

    ready = False
    for _ in range(60):
        for url in [f"{SERVER_URL}/api/games", f"{SERVER_URL}/games", f"{SERVER_URL}/"]:
            try:
                r = requests.get(url, headers={"X-API-Key": "local"}, timeout=2)
                if r.status_code < 500:
                    ready = True
                    break
            except Exception:
                pass
        if ready:
            break
        time.sleep(1)
    if not ready:
        raise RuntimeError("Local competition server failed to start.")
    return local_arc, thread


def replay_plan(arcade, game_id, plan):
    env = arcade.make(game_id=game_id, seed=0, save_recording=False)
    obs = None
    executed = 0
    for act in plan:
        try:
            obs = env.step(act.enum(), data=act.payload(), reasoning={"solver": "src_bfs"})
        except TypeError:
            obs = env.step(act.enum(), act.payload())
        executed += 1
    return executed, obs


def guess_score_column(df):
    for cand in ["score", "game_score", "rhae", "avg_score"]:
        if cand in df.columns:
            return cand
    for col in df.columns:
        if "score" in str(col).lower():
            return col
    numeric = list(df.select_dtypes(include=[np.number]).columns)
    if numeric:
        ranked = []
        for col in numeric:
            s = df[col].dropna()
            if len(s) == 0:
                continue
            lo = float(s.min()); hi = float(s.max())
            penalty = 0 if (0.0 <= lo <= 100.0 and 0.0 <= hi <= 100.0) else 5
            ranked.append((penalty, s.nunique(), col))
        if ranked:
            ranked.sort()
            return ranked[0][2]
    return None


def wait_for_submission():
    p = WORK_DIR / "submission.parquet"
    for _ in range(60):
        if p.exists():
            return p
        time.sleep(1)
    return p


game_infos = scan_games(ENV_DIR)
logger.info(f"Discovered {len(game_infos)} games under {ENV_DIR}")

local_server, local_thread = start_local_server()
logger.info("Local competition server is up.")

competition_arcade = Arcade(
    operation_mode=OperationMode.COMPETITION,
    arc_api_key="local",
    arc_base_url=SERVER_URL,
    environments_dir=str(ENV_DIR),
)

search_deadline = START_TIME + SEARCH_BUDGET_SECONDS
debug_rows = []
executed_anything = False
replayed_games = 0

total_games = len(game_infos)
for idx, info in enumerate(game_infos, start=1):
    if time.time() >= search_deadline or remaining_total_time() <= 120:
        logger.info("Stopping search due to notebook time budget.")
        break

    remaining_games = max(1, total_games - idx + 1)
    remaining_search = max(30.0, search_deadline - time.time())
    if total_games <= 150:
        cap = 120.0
    elif total_games <= 400:
        cap = 60.0
    else:
        cap = 25.0
    budget = min(cap, max(4.0, remaining_search / remaining_games * 1.15))
    if "ACTION6" not in info.source_code:
        budget *= 1.15
    if len(info.source_code) < 5000:
        budget *= 1.10
    budget = min(cap, budget)

    logger.info(f"[{idx}/{total_games}] {info.game_id}: budget={budget:.1f}s class={info.class_name}")
    t0 = time.time()
    plan = []
    solved_levels = 0
    replayed_actions = 0
    replay_status = "skipped"

    try:
        loader = GameLoader(info)
        loader.load()
        solver = Solver(info, loader, deadline=time.time() + budget, scan_timeout=min(4.0, max(1.0, budget * 0.10)))
        plan, solved_levels = solver.solve_game()
    except Exception as e:
        logger.warning(f"{info.game_id}: solver crashed: {e}")
        logger.debug(traceback.format_exc())

    if plan:
        try:
            replayed_actions, _ = replay_plan(competition_arcade, info.game_id, plan)
            replay_status = "replayed"
            replayed_games += 1
            executed_anything = executed_anything or (replayed_actions > 0)
        except Exception as e:
            replay_status = f"replay_failed:{type(e).__name__}"
            logger.warning(f"{info.game_id}: replay failed: {e}")

    elapsed = time.time() - t0
    debug_rows.append({
        "game_id": info.game_id,
        "title": info.title,
        "budget_sec": round(budget, 3),
        "elapsed_sec": round(elapsed, 3),
        "levels_solved_offline": solved_levels,
        "planned_actions": len(plan),
        "replayed_actions": replayed_actions,
        "replay_status": replay_status,
        "source_len": len(info.source_code),
        "has_action6": "ACTION6" in info.source_code,
    })
    pd.DataFrame(debug_rows).to_csv(WORK_DIR / "solver_debug.csv", index=False)
    gc.collect()

if not executed_anything and game_infos:
    logger.info("No solved plan found; sending one minimal action to force submission generation.")
    try:
        env = competition_arcade.make(game_id=game_infos[0].game_id, seed=0, save_recording=False)
        if ALL_SIMPLE:
            env.step(getattr(GameAction, ALL_SIMPLE[0]), reasoning={"solver": "fallback"})
            executed_anything = True
    except Exception as e:
        logger.warning(f"Fallback action failed: {e}")

try:
    _closed = competition_arcade.close_scorecard()
except Exception as e:
    logger.warning(f"close_scorecard failed: {e}")

submission_path = wait_for_submission()
submission = pd.read_parquet(submission_path) if submission_path.exists() else pd.DataFrame()
debug_df = pd.DataFrame(debug_rows)
debug_df.to_csv(WORK_DIR / "solver_debug.csv", index=False)

score_col = guess_score_column(submission) if not submission.empty else None
avg25 = float(submission[score_col].head(25).mean()) if (score_col is not None and not submission.empty) else float("nan")

print("games_scanned =", len(game_infos))
print("games_replayed =", replayed_games)
print("submission_path =", submission_path)
print("submission_exists =", submission_path.exists())
print("submission_rows =", 0 if submission.empty else len(submission))
print("score_column =", score_col)
print("avg_first_25 =", avg25)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

submission_path = Path("/kaggle/working/submission.parquet")
submission = pd.read_parquet(submission_path) if submission_path.exists() else pd.DataFrame()

print(submission.head(25))

score_col = None
for cand in ["score", "game_score", "rhae", "avg_score"]:
    if cand in submission.columns:
        score_col = cand
        break
if score_col is None:
    for col in submission.columns:
        if "score" in str(col).lower():
            score_col = col
            break
if score_col is None:
    numeric_cols = list(submission.select_dtypes(include=[np.number]).columns)
    if numeric_cols:
        ranked = []
        for col in numeric_cols:
            s = submission[col].dropna()
            if len(s) == 0:
                continue
            lo = float(s.min())
            hi = float(s.max())
            penalty = 0 if (0.0 <= lo <= 100.0 and 0.0 <= hi <= 100.0) else 5
            ranked.append((penalty, s.nunique(), col))
        if ranked:
            ranked.sort()
            score_col = ranked[0][2]

if score_col is None or submission.empty:
    print("Average score of first 25 games: NaN")
else:
    avg_first_25 = float(submission[score_col].head(25).mean())
    print(f"Average score of first 25 games ({score_col}): {avg_first_25}")